# BBC News Text Classification using Naive Bayes

### Import Libraries

In [1]:
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

### Data Loading

In [2]:
url = "https://raw.githubusercontent.com/ronaldleonardo/Data_Transformer_Datasets/refs/heads/main/bbc_text_cls.csv"

In [3]:
df = pd.read_csv(url)

In [4]:
df

,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business
...,...,...
2219,BT program to beat dialler scams\n\nBT is intr...,tech
2220,Spam e-mails tempt net shoppers\n\nComputer us...,tech
2221,Be careful how you code\n\nA new European dire...,tech
2222,US cyber security chief resigns\n\nThe man mak...,tech


### Exploratory Data Analysis

### Text Preprocessing
- Lowercasing
- Removing punctuation
- Stopword removal
- Stemming

In [5]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\Shreya R
[nltk_data]     Joshi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Shreya R
[nltk_data]     Joshi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [6]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

sw = set(stopwords.words('english'))

ps = PorterStemmer()
lemma = WordNetLemmatizer()

def clean(review, technique=None):
    
    # Step-1 Lowercase
    review = review.lower()
    
    # Step-2 Remove punctuation
    review = re.sub(r"[^a-zA-Z\s]", "", review)
    
    # Step-3 Remove stopwords
    review = [i for i in review.split() if i not in sw]
    
    # Step-4 Stemming
    if technique == "stem":
        review = [ps.stem(i) for i in review]
    
    # Step-5 Lemmatization
    elif technique == "lemma":
        review = [lemma.lemmatize(i) for i in review]
    
    # Convert list back to sentence
    review = " ".join(review)
    
    return review

In [7]:
df["cleaned_text"] = df["text"].apply(clean)

In [8]:
df.head()

,text,labels,cleaned_text
0,Ad sales boost Time Warner profit\n\nQuarterly...,business,ad sales boost time warner profit quarterly pr...
1,Dollar gains on Greenspan speech\n\nThe dollar...,business,dollar gains greenspan speech dollar hit highe...
2,Yukos unit buyer faces loan claim\n\nThe owner...,business,yukos unit buyer faces loan claim owners embat...
3,High fuel prices hit BA's profits\n\nBritish A...,business,high fuel prices hit bas profits british airwa...
4,Pernod takeover talk lifts Domecq\n\nShares in...,business,pernod takeover talk lifts domecq shares uk dr...


### Feature Extraction (TF-IDF)

In [9]:
tfidf = TfidfVectorizer()
tfidf

TfidfVectorizer()

In [10]:
tfidf.fit_transform(df["cleaned_text"]).toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

### Train Test Split

In [11]:
X = df["cleaned_text"]
y = df["labels"]

In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [13]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((1779,), (445,), (1779,), (445,))

In [14]:
X_train_transformed = tfidf.fit_transform(X_train).toarray()
X_test_transformed = tfidf.transform(X_test).toarray()

### Model Building using Multinomial Naive Bayes

In [15]:
from sklearn.naive_bayes import MultinomialNB

In [16]:
mnb = MultinomialNB(alpha=1.0)

In [17]:
mnb.fit(X_train_transformed,y_train)

MultinomialNB()

In [18]:
y_pred_mnb = mnb.predict(X_test_transformed)


### Model Evaluation

In [19]:
accuracy_score(y_test,y_pred_mnb)

0.9640449438202248

In [20]:
confusion_matrix(y_test,y_pred_mnb)

array([[108,   0,   3,   0,   2],
       [  2,  73,   2,   1,   0],
       [  1,   0,  66,   0,   2],
       [  1,   0,   0, 100,   0],
       [  0,   1,   0,   1,  82]], dtype=int64)

In [21]:
print(classification_report(y_test,y_pred_mnb))

               precision    recall  f1-score   support

     business       0.96      0.96      0.96       113
entertainment       0.99      0.94      0.96        78
     politics       0.93      0.96      0.94        69
        sport       0.98      0.99      0.99       101
         tech       0.95      0.98      0.96        84

     accuracy                           0.96       445
    macro avg       0.96      0.96      0.96       445
 weighted avg       0.96      0.96      0.96       445



### Prediction on Custom Input

In [22]:
text ="Indian equity benchmark indices settled slightly lower on Thursday led by muted global cues as the profit booking in the select pockets capped the gains .  The BSE Sensex fell 114 points, or 0.15 per cent, to close at 77,844.52, while NSE's Nifty50 shed 4.30 points, or 0.02 per cent, to end at 24,326.65 for the day. Here are the stocks that may remain under spotlight before the opening bell on Friday, May 08, 2026:"

In [23]:
pd.Series(clean(text))

0    indian equity benchmark indices settled slight...
dtype: object

In [24]:
text_trans = tfidf.transform(pd.Series(clean(text))).toarray()

In [25]:
mnb.predict(text_trans)

array(['business'], dtype='<U13')

## Conclusion

- The Multinomial Naive Bayes model performed well for BBC news text classification using TF-IDF features. 
- The preprocessing pipeline improved text quality and helped the model classify news categories effectively.

In [26]:
tfidf
mnb

MultinomialNB()

In [27]:
import pickle

# Save both the TF-IDF vectorizer and the MNB model
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("mnb_model.pkl", "wb") as f:
    pickle.dump(mnb, f)

print("✅ Model and Vectorizer saved successfully!")

✅ Model and Vectorizer saved successfully!
